## Initialization

In [ ]:
# Imports
# import pickle
# from pathlib import Path
from typing import Callable, Literal
from random import sample

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from scipy.signal import find_peaks
from scipy.interpolate import CubicSpline
from scipy.optimize import curve_fit
import pandas as pd
# from scipy.interpolate import interp1d
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.figure_of_merit import gaussian
# from data_processing.processing.neutron_classification import classify
# from data_processing.processing.neutron_window_generation import (
#     generate_nasa_neutron_window,
#     generate_n_distro_neutron_window
# )
from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
# from data_processing.arc_paths import (INPUT_DATA_FOLDER, get_exp_root,
#                                        get_parq_root)
from data_processing.dataframe_validation import DetectorDataframeColumn, EnergyColumn
from data_processing.experiment_data_keys import (ExperimentDataKey,
                                                  ExperimentNeutronData)
from data_processing.helpers import (
    get_input_with_default,
    # get_input_required,
    input_experiment_ids,
    stop
)
from data_processing.reporting.plotting import plot_scatter
# from data_processing.loading import get_neutron_window_paths, load_side_borders
# from data_processing.loading.dataframe_loading import load_psd
# from data_processing.loading.timetag_processing import calculate_timetag_hours
# from data_processing.reporting import plot_classification
# from data_processing.types import (BimodalBounds, BimodalParams,
#                                    NasaGenerationSettings,
#                                    NeutronWindowSettings, WindowType)
# from scipy.optimize import curve_fit
# from scipy.signal import deconvolve
from data_processing.types import NasaGenerationSettings
from data_processing import helpers

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

### Functions

In [ ]:
def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], proc.AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], proc.AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))

## Experiment ID Input

In [ ]:
# experiment_ids = input_experiment_ids()
experiment_ids = ["TB-26"]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

## Data Loading and Initial Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id, with_flags=True)
    exp_data["signals_df"] = load.load_parquet_signals(exp_id)

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = load.calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, proc.Detector.ZERO)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Neutron Classification

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 20e-3

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = proc.get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_width=energy_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    # # Default
    # default_bounds: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
    # )

    # bounds_a: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
    # )

    # bounds_b: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
    # )

    # # Ranged Example
    # bounds = [
    #     ((0, 60), bounds_a),
    # ]

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        # default_bounds,
        # bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = proc.classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

## Pulse Height Distribution

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    signals_df = exp_data["signals_df"].astype("int32")

    signals_np = signals_df.to_numpy()
    baselines = signals_np[:, :30].mean(axis=1).reshape(-1, 1)
    signals_np = -signals_np + baselines
    signals_df = pd.DataFrame(signals_np, index=signals_df.index, columns=signals_df.columns)
    psd_report["peak_height"] = signals_df.max(axis=1)
    # print(neutrons_only["peak_height"].max())
    # print(neutrons_only["peak_height"].min())

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report
    exp_data["signals_df"] = signals_df

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    signals_df = exp_data["signals_df"]
    n_class_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    
    gamma_only = psd_report.query(f"~{n_class_col_name}").copy()
    neutrons_only = psd_report.query(n_class_col_name).copy()
    n_signals_df = signals_df.loc[neutrons_only.index]
    g_signals_df = signals_df.loc[gamma_only.index]

    print(neutrons_only['peak_height'].min(), neutrons_only['peak_height'].max())
    print(gamma_only['peak_height'].min(), gamma_only['peak_height'].max())
    
    exp_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    exp_data[ExperimentDataKey.GAMMA_ONLY] = gamma_only
    exp_data["n_signals_df"] = n_signals_df
    exp_data["g_signals_df"] = g_signals_df

In [ ]:
data_set_key = "all"
exp_data_key = ExperimentDataKey.PSD_REPORT

# data_set_key = "neutrons"
# exp_data_key = ExperimentDataKey.NEUTRONS_ONLY

# data_set_key = "gamma"
# exp_data_key = ExperimentDataKey.GAMMA_ONLY

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    df = exp_data[exp_data_key]
    peak_height = df["peak_height"]
    print(peak_height.min(), peak_height.max())

    bins_min = 0
    bins_max = 16000
    bins_step = 200
    bins = np.arange(bins_min, bins_max+bins_step, step=bins_step)
    # print(bins)
    
    Z_n, *_ = np.histogram(peak_height, bins=bins)
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = {
        data_set_key: {"standard": Z_n, "bins": bins},
    }

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    df = exp_data[exp_data_key]
    pulse_energy = df[DetectorDataframeColumn.ENERGY.value]
    print(pulse_energy.min(), pulse_energy.max())

    bins_min = 0
    bins_max = 4000
    bins_step = 50
    bins = np.arange(bins_min, bins_max+bins_step, step=bins_step)
    # print(bins)
    
    Z_n, *_ = np.histogram(pulse_energy, bins=bins)
    exp_data["pulse_energy_distribution"] = {
        data_set_key: {"standard": Z_n, "bins": bins},
    }

In [ ]:
# moving average
for exp_id, exp_data in experiment_neutron_data.items():
    phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
    phd_n_histogram = phd_histogram_data[data_set_key]["standard"]

    phd_n_moving_average = moving_average_centered(phd_n_histogram)

    phd_histogram_data[data_set_key]["moving_average"] = phd_n_moving_average
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = phd_histogram_data

In [ ]:
# moving average
for exp_id, exp_data in experiment_neutron_data.items():
    ped_histogram_data = exp_data["pulse_energy_distribution"]
    ped_n_histogram = ped_histogram_data[data_set_key]["standard"]

    ped_n_moving_average = moving_average_centered(ped_n_histogram)

    ped_histogram_data[data_set_key]["moving_average"] = ped_n_moving_average
    exp_data["pulse_energy_distribution"] = ped_histogram_data

## 2D Histograms

In [ ]:
adc_width = 20
psd_bin_count = 100
psd_min = 0
psd_max = 0.5

for exp_id, exp_data in experiment_neutron_data.items():
    df = exp_data[exp_data_key]
    x = df["peak_height"]
    y = df["tail / total"]

    within_psd = y.between(psd_min, psd_max)
    x = x[within_psd == True].copy()
    y = y[within_psd == True].copy()

    x_bins = np.linspace(0, x.max(), int(x.max() / adc_width) + 1)
    print(f"Energy width = {x_bins[1]-x_bins[0]} ADC")
    y_bins: np.ndarray = np.linspace(psd_min, psd_max, psd_bin_count + 1)

    Z, xe, ye = np.histogram2d(x, y, bins=[x_bins, y_bins])
    histo_data = {"counts": Z, "x_edges": xe, "y_edges": ye}
    exp_data["height_histo2d"] = histo_data

In [ ]:
adc_width = 20
psd_bin_count = 100
psd_min = 0
psd_max = 0.5

for exp_id, exp_data in experiment_neutron_data.items():
    df = exp_data[exp_data_key]
    x = df["ENERGY"]
    y = df["tail / total"]

    within_psd = y.between(psd_min, psd_max)
    x = x[within_psd == True].copy()
    y = y[within_psd == True].copy()

    x_bins = np.linspace(0, x.max(), int(x.max() / adc_width) + 1)
    print(f"Energy width = {x_bins[1]-x_bins[0]} ADC")
    y_bins: np.ndarray = np.linspace(psd_min, psd_max, psd_bin_count + 1)

    Z, xe, ye = np.histogram2d(x, y, bins=[x_bins, y_bins])
    histo_data = {"counts": Z, "x_edges": xe, "y_edges": ye}
    exp_data["energy_histo2d"] = histo_data

## Curve Fitting

In [ ]:
def power_fn(x, k, p):
    return k * np.pow(x, p)

In [ ]:
def linear(x, m, b):
    return m * x + b

In [ ]:
fit_fn = linear
# fit_fn = power_fn

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[exp_data_key]
    peak_height = psd_report["peak_height"]
    pulse_energy = psd_report[DetectorDataframeColumn.ENERGY.value]

    fit_params, _ = curve_fit(fit_fn, peak_height, pulse_energy)
    residuals = pulse_energy - fit_fn(peak_height, *fit_params)
    ss_res = np.sum(residuals**2)
    y_variance = pulse_energy - np.mean(pulse_energy)
    ss_tot = np.sum(y_variance**2)
    r_squared = 1 - (ss_res / ss_tot)
    print(fit_params, r_squared)
    if "height_energy_fit" not in exp_data:
        exp_data["height_energy_fit"] = {}
    exp_data["height_energy_fit"][data_set_key] = fit_params

## Display

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    # psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION][data_set_key]
    ped_histogram_data = exp_data["pulse_energy_distribution"][data_set_key]
    # selected_traces = exp_data["selected_traces"]
    height_bins = phd_histogram_data["bins"]
    phd_histogram = phd_histogram_data["standard"]
    phd_moving_avg = phd_histogram_data["moving_average"]
    phd_moving_avg = np.nan_to_num(phd_moving_avg)

    energy_bins = ped_histogram_data["bins"]
    ped_histogram = ped_histogram_data["standard"]
    ped_moving_avg = ped_histogram_data["moving_average"]
    ped_moving_avg = np.nan_to_num(ped_moving_avg)

    height_bin_mids = (height_bins[1:] + height_bins[:-1]) / 2
    energy_bin_mids = (energy_bins[1:] + energy_bins[:-1]) / 2
    # energy_bin_mids = energy_bin_mids * (15000 / 4000)

    # phd_spline = CubicSpline(energy_bin_mids, phd_n_moving_avg)
    # phd_spline_x = np.linspace(0, energy_bins.max(), num=1000)
    # phd_spline_y = phd_spline(phd_spline_x)

    fig, axs = plt.subplots(figsize=(12, 8))
    # fig.subplots_adjust(wspace=0)
    # ax1, ax2 = axs
    ax1 = axs
    ax2 = axs.twiny()

    ax1.plot(height_bin_mids, phd_histogram, color=bg_blue, lw=3)
    ax2.plot(energy_bin_mids, ped_histogram, color=bg_grey, lw=3)
    # ax1.plot(phd_spline_y, phd_spline_x, color="black")
    # ax1.plot(phd_n_moving_avg, energy_bin_mids, color=bg_blue, linewidth=3)
    # ax1.fill_betweenx(energy_bin_mids, 0, phd_n_moving_avg, color=bg_grey)
    # for i, (bin_mid, trace) in enumerate(selected_traces):
    #     trace_x = [x + i * 25 for x in range(len(trace))]
    #     ax2.plot(trace_x, trace, label=bin_mid)

    ax1.set_yscale("log")
    # ax1.xaxis.set_inverted(True)
    # ax1.xaxis.set_ticks_position("top")
    # ax1.xaxis.set_label_position("top")
    # ax1.yaxis.set_inverted(True)
    # ax1.xaxis.set_tick_params(labelbottom=False)
    # ax1.yaxis.set_tick_params(labelbottom=False, bottom=False)
    # ax2.xaxis.set_tick_params(labelbottom=False)
    # ax2.yaxis.set_tick_params(labelbottom=False, direction="in")
    
    # ax1.spines["top"].set_visible(False)
    # ax1.spines["left"].set_visible(False)
    # ax2.spines["top"].set_visible(False)
    # ax2.spines["right"].set_visible(False)

    # ax1.set_xlabel("Pulse height (ADC channel x1000)", fontsize=fontsize)
    ax1.set_xlabel("Pulse height (ADC channel)", fontsize=fontsize)
    ax2.set_xlabel("Pulse energy (ADC channel)", fontsize=fontsize)
    ax1.set_ylabel("Counts (x1000)", fontsize=fontsize)
    ax1.xaxis.set_major_formatter(lambda x, pos: f"{x/1000:.1f}")
    # ax1.yaxis.set_major_formatter(lambda x, pos: f"{x/1000:.1f}")
    ax1.tick_params(labelsize=fontsize)
    ax2.tick_params(labelsize=fontsize)
    # ax2.set_xlabel("Time (ns)", fontsize=fontsize)

    # limits = (0, 5000)
    # ax1.set_ylim(*limits)
    # ax2.set_ylim(*limits)
    # ax1.set_xlim(0, None)
    # ax2.set_xlim(None, 200)

    # for i, (bin_mid, trace) in enumerate(selected_traces):
    #     bin_mid_idx = np.where(energy_bin_mids == bin_mid)[0]
    #     bin_lo = float(energy_bins[bin_mid_idx][0])
    #     bin_hi = float(energy_bins[bin_mid_idx+1][0])
    #     bin_phd_x = np.linspace(bin_lo, bin_hi).reshape(-1, 1)
    #     bin_phd_y = phd_spline(bin_phd_x).reshape(-1, 1)
    #     bin_phd_xy = np.concatenate((bin_phd_y, bin_phd_x), axis=1)
    #     trace_max_x = float(trace.idxmax()) + (i * 25)
    #     bin_trace_xy = [[trace_max_x, bin_hi], [trace_max_x, bin_lo]]
        
    #     ax1_to_display = ax1.transData.transform
    #     ax2_to_display = ax2.transData.transform
    #     display_to_figure = fig.transFigure.inverted().transform

    #     bin_phd_xy = display_to_figure(ax1_to_display(bin_phd_xy))
    #     bin_trace_xy = display_to_figure(ax2_to_display(bin_trace_xy))
    #     bin_xy = np.concatenate((bin_phd_xy, bin_trace_xy))
        
    #     poly = mpl.patches.Polygon(bin_xy, closed=True, color="lightblue", alpha=0.3)
    #     fig.add_artist(poly)

    # get bin midpoints
    # make subplots (2 cols, 1 row, merged y-axis, no borders)
    # plot PHD histogram on left subplot (rotated)
    # pick 5 evenly spaced bins
    # for each bin, find first trace within that bin
    # (alternate, scan bin for trace with energy closest to midpoint of bin)
    # plot 5 traces on right subplot
    # make connecting lines between left subplot (bin midpoint, bin count) and right subplot (bin midpoint, trace height)

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[exp_data_key]
    fit_params = exp_data["height_energy_fit"][data_set_key]
    peak_height = psd_report["peak_height"]
    pulse_energy = psd_report[DetectorDataframeColumn.ENERGY.value]

    # fit_k, fit_p, *_ = fit_params
    fit_x = np.linspace(0, 15000, 200)
    fit_y = fit_fn(fit_x, *fit_params)

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.scatter(peak_height, pulse_energy, s=2, c=bg_blue, marker="o")
    ax.plot(fit_x, fit_y, color=bg_grey, lw=3)

    ax.set_xlim(0, 15000)
    ax.set_ylim(0, 4000)
    ax.set_xlabel("Pulse height (ADC channel x1000)", fontsize=fontsize)
    ax.set_ylabel("Pulse energy (ADC channel x1000)", fontsize=fontsize)
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x/1000:.1f}")
    ax.yaxis.set_major_formatter(lambda x, pos: f"{x/1000:.1f}")
    ax.tick_params(labelsize=fontsize)

In [ ]:
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]
    # print(psd_report.head())
    fig, ax = plot_scatter(
        psd_report["ENERGY"],
        psd_report[DetectorDataframeColumn.PSD.value]
    )
    ax.set_xlabel("Energy [ADC channels]", fontsize=14)
    ax.set_ylabel("PSD", fontsize=14)
    
    ax.set_title(
        f"{exp_name} PSD/Energy Graph",
        ha='center',
        fontsize=20
    )
    # output_path = get_report_root(exp_name) / f"{exp_name} PSD graph.png"
    # from pathlib import Path
    # output_path = Path() / f"{exp_name} PSD graph.png"
    # print(str(output_path))
    # fig.savefig(str(output_path))
    plt.show()

In [ ]:
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]
    fig, ax = plot_scatter(
        psd_report["peak_height"],
        psd_report[DetectorDataframeColumn.PSD.value]
    )
    ax.set_xlabel("Peak height [ADC channels]", fontsize=14)  # Update x-axis label
    ax.set_ylabel("PSD", fontsize=14)
    
    ax.set_title(
        f"{exp_name} PSD/Height Graph",
        ha='center',
        fontsize=20
    )
    # output_path = get_report_root(exp_name) / f"{exp_name} PSD graph.png"
    # from pathlib import Path
    # output_path = Path() / f"{exp_name} PSD graph.png"
    # print(str(output_path))
    # fig.savefig(str(output_path))
    plt.show()

In [ ]:
# histogram contour plot (vaporwave island)
cmap = plt.colormaps["nipy_spectral"]
figsize = (12, 12)
fontsize = 16
histo_res = 128
contour_res = 100
angle_elev = 30
angle_rot = -20

for exp_name, exp_data in experiment_neutron_data.items():
    data_dict = exp_data["height_histo2d"]
    Z = data_dict["counts"]
    xe = data_dict["x_edges"]
    ye = data_dict["y_edges"]
    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection='3d')
    x, y = np.meshgrid(xe[:-1], ye[:-1])

    ax.view_init(angle_elev, angle_rot)
    ax.contour3D(x, y, Z.T, contour_res, cmap=cmap)
    ax.set_title(f"{exp_name} PSD/Pulse height 3D Histogram", fontsize=fontsize+4)
    ax.set_ylabel("PSD", fontsize=fontsize)
    ax.set_xlabel("Peak height (MeVee)", fontsize=fontsize)
    ax.set_zlabel("Counts", fontsize=fontsize)

    # output_path = (
    #     # get_report_root(exp_name)
    #     Path()
    #     / f"{exp_name} PSD 3D Histogram.png"
    # )
    # fig.savefig(output_path)

    plt.show()

In [ ]:
# histogram contour plot (vaporwave island)
cmap = plt.colormaps["nipy_spectral"]
figsize = (12, 12)
fontsize = 16
histo_res = 128
contour_res = 100
angle_elev = 30
angle_rot = -20

for exp_name, exp_data in experiment_neutron_data.items():
    data_dict = exp_data["energy_histo2d"]
    Z = data_dict["counts"]
    xe = data_dict["x_edges"]
    ye = data_dict["y_edges"]
    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection='3d')
    x, y = np.meshgrid(xe[:-1], ye[:-1])

    ax.view_init(angle_elev, angle_rot)
    ax.contour3D(x, y, Z.T, contour_res, cmap=cmap)
    ax.set_title(f"{exp_name} PSD/Energy 3D Histogram", fontsize=fontsize+4)
    ax.set_ylabel("PSD", fontsize=fontsize)
    ax.set_xlabel("Pulse height (MeVee)", fontsize=fontsize)
    ax.set_zlabel("Counts", fontsize=fontsize)

    # output_path = (
    #     # get_report_root(exp_name)
    #     Path()
    #     / f"{exp_name} PSD 3D Histogram.png"
    # )
    # fig.savefig(output_path)

    plt.show()